# Experimento 01 - Indicadores Aislados (RSI, MACD, MA, CCI) - Long Only
> Pregunta: Existe evidencia de que RSI, MACD, MA o CCI tengan capacidad por si solos para generar estrategia rentable en BTC? Cada indicador se prueba aislado (no combinado). Solo LONG. Optimizacion TPE/NSGA-II + Walk-Forward + metricas completas.

```
BTC Historical Data (data/processed/btcusdt_1m.parquet 4.7M x 17  2017-08-17 -> 2026-09-01)
        | OHLCV
+---------------------------------+
| RSI  (mean-reversion)            |
| MACD (cruce senal)               |
| Moving Average  SMA vs EMA       |  <- aislados
| CCI  (Commodity Channel Index)    |
+---------------------------------+
        | Parameter Optimizer (TPE / NSGA-II)
    Backtester (long-only, lag=1, fees 0.10% + slippage 5bps)
        | Walk-Forward (IS 3y -> OOS 1y, HOLDOUT 2025-2026)
        | Risk + Return + Robustness Metrics (17 categorias)
        | Comparacion vs BTC Buy & Hold
```
> Stack: polars + numpy + duckdb + sklearn + xgboost/lightgbm + matplotlib + optuna (ver AGENTS.md y requirements.txt). Datos maestro 1m ya poblado.


## 1. Setup - imports, carga datos, timeframe y configs
> TIMEFRAME configurable: 1m (4.7M) pesado, 1h (79k) rapido para iterar. Cambiar TIMEFRAME="1m" para experimento final.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import polars as pl, numpy as np, pandas as pd
import datetime as dt
import yaml, pathlib, sys, math
from pathlib import Path
if r'E:\bitcoin-trading-research' not in sys.path:
    sys.path.insert(0, r'E:\bitcoin-trading-research')
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize']=(12,4)
plt.rcParams['axes.grid']=True
plt.rcParams['grid.alpha']=0.3
REPO = pathlib.Path(r'E:\bitcoin-trading-research')
if not (REPO / 'configs/data.yaml').exists():
    # fallback si cwd es notebooks/
    REPO = pathlib.Path.cwd()
    if not (REPO / 'configs/data.yaml').exists():
        REPO = REPO.parent
    if not (REPO / 'configs/data.yaml').exists():
        REPO = pathlib.Path(r'E:\bitcoin-trading-research')
print(f'REPO={REPO} exists configs={ (REPO / "configs/data.yaml").exists()}')
cfg_data=yaml.safe_load(open(REPO / 'configs/data.yaml', encoding='utf-8'))
cfg_strat=yaml.safe_load(open(REPO / 'configs/strategies.yaml', encoding='utf-8'))
print('symbol', cfg_data['binance']['symbol'], 'interval', cfg_data['binance']['interval'])
print('folds', cfg_data['validation']['folds'])
TIMEFRAME='4h'          # '1m' pesado (4.7M) | '1h' rapido (79k) | '4h' tesis (19k) - cambiado a 4h por peticion
FEES_BPS=10
SLIPPAGE_BPS=5
LAG=1
N_TRIALS_TPE=80
N_TRIALS_NSGA=80
SEED=42
print(f'TIMEFRAME={TIMEFRAME} FEES={FEES_BPS}bps SLIPPAGE={SLIPPAGE_BPS}bps')


In [ ]:
parquet_1m=REPO / 'data/processed/btcusdt_1m.parquet'
assert parquet_1m.exists(), 'Falta parquet'
df_1m=pl.read_parquet(parquet_1m)
print(f'1m: {df_1m.shape} {df_1m["timestamp"].min()} -> {df_1m["timestamp"].max()}')
def resample_ohlcv(df, tf):
    if tf=='1m': return df
    every={'15m':'15m','1h':'1h','4h':'4h','1d':'1d','5m':'5m'}[tf]
    out=(df.sort('timestamp').group_by_dynamic('timestamp', every=every).agg([pl.col('open').first().alias('open'),pl.col('high').max().alias('high'),pl.col('low').min().alias('low'),pl.col('close').last().alias('close'),pl.col('volume').sum().alias('volume'),pl.col('quote_volume').sum().alias('quote_volume'),pl.col('trade_count').sum().alias('trade_count'),pl.col('taker_buy_volume').sum().alias('taker_buy_volume'),pl.col('taker_buy_quote_volume').sum().alias('taker_buy_quote_volume'),pl.col('taker_sell_volume').sum().alias('taker_sell_volume'),pl.col('taker_sell_quote_volume').sum().alias('taker_sell_quote_volume')]).sort('timestamp').with_columns([(pl.col('taker_buy_volume')/pl.col('volume')).alias('buy_ratio'),(pl.col('taker_sell_volume')/pl.col('volume')).alias('sell_ratio'),(pl.col('taker_buy_volume')-pl.col('taker_sell_volume')).alias('volume_delta')]))
    return out
df=resample_ohlcv(df_1m, TIMEFRAME)
print(f'{TIMEFRAME}: {df.shape} {df["timestamp"].min()} -> {df["timestamp"].max()}')
df.head(3).to_pandas()


## 1b. Halvings BTC — contexto y analisis por ciclo
BTC halvings: `2012-11-28` (no en dataset), `2016-07-09`, `2020-05-11`, `2024-04-19`, `2028 ~`. Para nuestro rango `2017-08-17 → 2026-09-01` los relevantes son **2016-07-09** (cola), **2020-05-11** y **2024-04-19**. Veloso et al. (2025) muestran volatilidad decreciente entre halvings — evaluaremos si la estrategia es robusta entre ciclos.
Se grafican como líneas verticales en equity y se calculan métricas por ciclo (CAGR, Sharpe, MaxDD) para ver si el edge depende del ciclo.


In [ ]:
# --- Halvings (fechas oficiales) ---
HALVINGS = [
    ('2016-07-09', 'Halving 2'),
    ('2020-05-11', 'Halving 3'),
    ('2024-04-19', 'Halving 4'),
]
# Convertir a datetime UTC para filtrar
HALVINGS_DT = [(pl.lit(d).str.to_datetime(time_zone='UTC').alias('dt')) for d,_ in HALVINGS]  # placeholder, usamos dt.datetime
import datetime as _dt
HALVINGS_DT = [(_dt.datetime.fromisoformat(d).replace(tzinfo=_dt.timezone.utc), label) for d,label in HALVINGS]
print('Halvings considerados:', HALVINGS)
# Helper para graficar halvings en eje temporal
def add_halvings_to_ax(ax):
    for d,label in HALVINGS_DT:
        # solo dibujar si esta dentro del rango de df
        if df['timestamp'].min() <= d <= df['timestamp'].max():
            ax.axvline(d, color='purple', ls='--', lw=1.0, alpha=0.7)
            ax.text(d, ax.get_ylim()[1]*0.95, label, rotation=90, va='top', ha='right', fontsize=7, color='purple')
    return ax
# Demo: plot close con halvings
import matplotlib.pyplot as plt
sample = df.tail(2000 if TIMEFRAME=='4h' else 720).to_pandas()
fig, ax = plt.subplots(figsize=(12,3))
ax.plot(sample['timestamp'], sample['close'], lw=0.8)
ax.set_title(f'BTCUSDT {TIMEFRAME} con halvings')
add_halvings_to_ax(ax)
plt.show()
# Helper metricas por ciclo: corta df por halving y calcula CAGR/Sharpe con summarize_bt
def metrics_by_halving_cycle(df_input, signal_fn, fee_bps=FEES_BPS, slippage_bps=SLIPPAGE_BPS):
    cycles = []
    # Ciclos: 2017-08-17..2020-05-11, 2020-05-11..2024-04-19, 2024-04-19..hoy
    bounds = [df['timestamp'].min()] + [d for d,_ in HALVINGS_DT if d > df['timestamp'].min() and d < df['timestamp'].max()] + [df['timestamp'].max()]
    for i in range(len(bounds)-1):
        start, end = bounds[i], bounds[i+1]
        sub = df_input.filter((pl.col('timestamp') >= start) & (pl.col('timestamp') < end))
        if len(sub) < 100: continue
        sig = signal_fn(sub)
        col = [c for c in sig.columns if c.startswith('signal')][0]
        bt = backtest_long(sig, col, fee_bps=fee_bps, slippage_bps=slippage_bps, lag=LAG)
        m = summarize_bt(bt)
        cycles.append({'cycle': f'{str(start)[:10]} -> {str(end)[:10]}', 'n_bars': len(sub), 'cagr': m['cagr'], 'sharpe': m['sharpe'], 'mdd': m['max_drawdown']})
    return cycles
print('Ejemplo metricas por ciclo (SMA 20/60):')
for r in metrics_by_halving_cycle(df, lambda d: signal_ma_long(d,20,60,'sma')):
    print(r)


In [ ]:
print(df.select(['open','high','low','close','volume','trade_count']).describe().to_pandas().to_string())
gaps=df.select((pl.col('timestamp').diff().dt.total_seconds()/3600).alias('gap_h')).filter(pl.col('gap_h')>1.5)
print(f'gaps >1.5h en {TIMEFRAME}: {len(gaps)}')
sample=df.tail(720 if TIMEFRAME=='1h' else 1000).to_pandas()
fig, ax=plt.subplots(figsize=(12,3))
ax.plot(sample['timestamp'], sample['close'], lw=0.8)
ax.set_title(f'BTCUSDT {TIMEFRAME} - ultimos {len(sample)} velas')
plt.show()


## 2. Definicion matematica de indicadores
RSI Wilder: RSI=100-100/(1+RS), RS=EMA(gain)/EMA(loss). MACD=EMA_fast-EMA_slow, Signal=EMA(MACD). SMA/EMA promedios. CCI: TP=(H+L+C)/3, CCI=(TP-SMA(TP))/(0.015*MeanDev). CCI alto no predice direccion, si magnitud futura.

In [ ]:
from src.features.technical import rsi, macd, cci, sma, ema
df_demo=df.clone()
df_demo=df_demo.with_columns(rsi(pl.col('close'),14))
ml, sl, h = macd(pl.col('close'),12,26,9)
df_demo=df_demo.with_columns([ml, sl, h])
df_demo=df_demo.with_columns([sma(pl.col('close'),20), sma(pl.col('close'),60), ema(pl.col('close'),20), ema(pl.col('close'),60)])
df_demo=df_demo.with_columns(cci(pl.col('high'), pl.col('low'), pl.col('close'),20))
print(df_demo.select(['timestamp','close','rsi_14','macd_line','macd_signal','sma_20','sma_60','cci_20']).tail(3).to_pandas().to_string())
d = df_demo.tail(500).to_pandas()
fig, axs = plt.subplots(4,1, figsize=(12,8), sharex=True)
axs[0].plot(d['timestamp'], d['close'], lw=0.7); axs[0].set_title('Close')
axs[1].plot(d['timestamp'], d['rsi_14'], lw=0.7); axs[1].axhline(70,c='r',ls='--',lw=0.7); axs[1].axhline(30,c='g',ls='--',lw=0.7); axs[1].set_title('RSI(14)')
axs[2].plot(d['timestamp'], d['macd_line'], lw=0.7, label='MACD'); axs[2].plot(d['timestamp'], d['macd_signal'], lw=0.7, label='Signal'); axs[2].legend(fontsize=7); axs[2].set_title('MACD 12,26,9')
axs[3].plot(d['timestamp'], d['cci_20'], lw=0.7); axs[3].axhline(100,c='r',ls='--',lw=0.7); axs[3].axhline(-100,c='g',ls='--',lw=0.7); axs[3].set_title('CCI(20)')
plt.tight_layout(); plt.show()


## 3. Reglas de trading - Long Only (aisladas)
No RSI+MACD+MA. Solo LONG (0/1), lag=1, fees 0.10%+5bps.

**RSI:** ENTRY RSI_{t-1}<OS and RSI_t>OS -> LONG, EXIT RSI>=50 -> FLAT. Rango period 2-50, OS 10-45, OB 55-90, OS<50<OB. Salida fija 50.

**MACD:** LONG si MACD cruza arriba Signal, FLAT si cruza abajo. Fast 3-30, Slow 15-100, Signal 2-30, Fast<Slow.

**MA:** SMA fast>slow -> LONG else FLAT; EMA igual pero experimento separado. Fast 2-100, Slow 20-400, Fast<Slow.

**CCI:** ENTRY CCI_{t-1}<entry and CCI_t>entry -> LONG, EXIT CCI>exit -> FLAT. period 5-50, entry -150..0, exit 0..150, entry<exit. Clasico -100/100 baseline. CCI alto predice |Return| futuro, evaluar como filtro.

In [ ]:
def signal_rsi_long(df, period, oversold, overbought, exit_level=50):
    assert oversold < 50 < overbought
    df=df.sort('timestamp')
    col=f'rsi_{period}'
    if col not in df.columns:
        df=df.with_columns(rsi(pl.col('close'), period))
    df=df.with_columns([pl.when((pl.col(col).shift(1) < oversold) & (pl.col(col) > oversold)).then(1).when(pl.col(col) >= exit_level).then(0).otherwise(None).alias('_sig')])
    df=df.with_columns(pl.col('_sig').forward_fill().fill_null(0).alias('signal_rsi'))
    return df.drop('_sig')
def signal_macd_long(df, fast, slow, signal):
    assert fast < slow
    df=df.sort('timestamp')
    ml, sl, h = macd(pl.col('close'), fast, slow, signal)
    df=df.with_columns([ml.alias('ml'), sl.alias('sl')])
    df=df.with_columns([pl.when((pl.col('ml').shift(1) < pl.col('sl').shift(1)) & (pl.col('ml') > pl.col('sl'))).then(1).when((pl.col('ml').shift(1) > pl.col('sl').shift(1)) & (pl.col('ml') < pl.col('sl'))).then(0).otherwise(None).alias('_sig')])
    df=df.with_columns(pl.col('_sig').forward_fill().fill_null(0).alias('signal_macd'))
    return df.drop(['ml','sl','_sig'])
def signal_ma_long(df, fast, slow, kind='sma'):
    assert fast < slow
    df=df.sort('timestamp')
    if kind=='sma':
        df=df.with_columns([sma(pl.col('close'), fast).alias('_fast'), sma(pl.col('close'), slow).alias('_slow')])
        col='signal_sma'
    else:
        df=df.with_columns([ema(pl.col('close'), fast).alias('_fast'), ema(pl.col('close'), slow).alias('_slow')])
        col='signal_ema'
    df=df.with_columns((pl.col('_fast') > pl.col('_slow')).cast(pl.Int8).alias(col))
    df=df.with_columns(pl.col(col).fill_null(0).alias(col))
    return df.drop(['_fast','_slow'])
def signal_cci_long(df, period, entry, exit_):
    assert entry < exit_
    df=df.sort('timestamp')
    col=f'cci_{period}'
    if col not in df.columns:
        df=df.with_columns(cci(pl.col('high'), pl.col('low'), pl.col('close'), period))
    df=df.with_columns([pl.when((pl.col(col).shift(1) < entry) & (pl.col(col) > entry)).then(1).when(pl.col(col) > exit_).then(0).otherwise(None).alias('_sig')])
    df=df.with_columns(pl.col('_sig').forward_fill().fill_null(0).alias('signal_cci'))
    return df.drop('_sig')
for fn, name in [(lambda d: signal_rsi_long(d,14,30,70), 'RSI 14/30/70'), (lambda d: signal_macd_long(d,12,26,9), 'MACD 12,26,9'), (lambda d: signal_ma_long(d,20,60,'sma'), 'SMA 20/60'), (lambda d: signal_ma_long(d,20,60,'ema'), 'EMA 20/60'), (lambda d: signal_cci_long(d,20,-100,100), 'CCI 20/-100/100')]:
    tmp=fn(df.head(1000))
    col=[c for c in tmp.columns if c.startswith('signal')][0]
    print(f"{name:15s} -> cambios={tmp[col].diff().abs().sum()} | %long={tmp[col].mean():.1%}")


## 4. Backtester Long-Only (sin look-ahead)
position_t = signal_{t-lag}, ret_t=close_t/close_{t-1}-1, turnover=|position_t-position_{t-1}|, cost=turnover*(fees+slippage)/10000, strat_ret=position*ret - cost, equity=capital*prod(1+strat_ret)

In [ ]:
from src.backtesting.metrics import sharpe, sortino, calmar, cagr, max_drawdown, win_rate, profit_factor
def backtest_long(df, signal_col, price_col='close', initial_capital=10000, fee_bps=10, slippage_bps=5, lag=1):
    df=df.sort('timestamp')
    df=df.with_columns(pl.col(signal_col).shift(lag).fill_null(0).alias('position'))
    df=df.with_columns((pl.col(price_col)/pl.col(price_col).shift(1)-1).alias('_ret'))
    df=df.with_columns((pl.col('position').diff().abs().fill_null(0)).alias('_turnover'))
    cost_rate=(fee_bps+slippage_bps)/10000
    df=df.with_columns((pl.col('_turnover')*cost_rate).alias('_cost'))
    df=df.with_columns((pl.col('position')*pl.col('_ret') - pl.col('_cost')).alias('strategy_ret'))
    df=df.with_columns((1+pl.col('strategy_ret').fill_null(0)).cum_prod().alias('_growth'))
    df=df.with_columns((pl.col('_growth')*initial_capital).alias('equity'))
    df=df.with_columns((pl.col(price_col)/pl.col(price_col).first()).alias('_bh_growth'))
    df=df.with_columns((pl.col('_bh_growth')*initial_capital).alias('bh_equity'))
    return df
def summarize_bt(bt):
    rets=bt['strategy_ret'].drop_nulls().to_numpy()
    eq=bt['equity'].drop_nulls().to_numpy()
    bh_eq=bt['bh_equity'].drop_nulls().to_numpy() if 'bh_equity' in bt.columns else None
    mdd, mdd_dur = max_drawdown(eq)
    total_ret=float(eq[-1]/eq[0]-1) if len(eq)>1 else 0.0
    bars_per_year={'1m':525600,'15m':35040,'1h':8760,'4h':2190,'1d':365}[TIMEFRAME]
    s=sharpe(rets, periods_per_year=bars_per_year)
    so=sortino(rets, periods_per_year=bars_per_year)
    cg=cagr(eq, periods_per_year=bars_per_year)
    ca=calmar(eq, periods_per_year=bars_per_year)
    wr=win_rate(rets)
    pf=profit_factor(rets)
    turnover=float(bt['_turnover'].sum()) if '_turnover' in bt.columns else 0.0
    fees_total=float(bt['_cost'].sum()*bt['equity'].first()) if '_cost' in bt.columns else 0.0
    exposure=float((bt['position']>0).mean())
    pd_rets=pd.Series(rets)
    skew=float(pd_rets.skew()) if len(rets)>2 else 0.0
    kurt=float(pd_rets.kurtosis()) if len(rets)>3 else 0.0
    gains=rets[rets>0]; losses=rets[rets<0]
    avg_win=float(gains.mean()) if len(gains)>0 else 0.0
    avg_loss=float(losses.mean()) if len(losses)>0 else 0.0
    payoff=float(abs(avg_win/avg_loss)) if avg_loss!=0 else 0.0
    expectancy=float(wr*avg_win - (1-wr)*abs(avg_loss)) if len(rets)>0 else 0.0
    bh_cagr=cagr(bh_eq, periods_per_year=bars_per_year) if bh_eq is not None else 0.0
    bh_sharpe=sharpe((bt['close'].pct_change().drop_nulls().to_numpy() if 'close' in bt.columns else rets), periods_per_year=bars_per_year) if bh_eq is not None else 0.0
    return {'total_return':total_ret,'cagr':cg,'sharpe':s,'sortino':so,'calmar':ca,'max_drawdown':mdd,'mdd_duration_bars':mdd_dur,'win_rate':wr,'profit_factor':pf,'avg_win':avg_win,'avg_loss':avg_loss,'payoff':payoff,'expectancy':expectancy,'turnover':turnover,'fees_total':fees_total,'exposure':exposure,'skew':skew,'kurtosis':kurt,'bh_cagr':bh_cagr,'bh_sharpe':bh_sharpe,'n_bars':len(bt),'final_equity':float(eq[-1]) if len(eq)>0 else 0.0}
rows=[]
for name, fn in [('RSI 14/30/70', lambda d: signal_rsi_long(d,14,30,70)), ('MACD 12,26,9', lambda d: signal_macd_long(d,12,26,9)), ('SMA 20/60', lambda d: signal_ma_long(d,20,60,'sma')), ('EMA 20/60', lambda d: signal_ma_long(d,20,60,'ema')), ('CCI 20/-100/100', lambda d: signal_cci_long(d,20,-100,100))]:
    sig=fn(df)
    col=[c for c in sig.columns if c.startswith('signal')][0]
    bt=backtest_long(sig, col, fee_bps=FEES_BPS, slippage_bps=SLIPPAGE_BPS, lag=LAG)
    m=summarize_bt(bt)
    m['strategy']=name
    rows.append(m)
    print(f"{name:15s} CAGR {m['cagr']:.1%} Sharpe {m['sharpe']:.2f} MaxDD {m['max_drawdown']:.1%} PF {m['profit_factor']:.2f} WR {m['win_rate']:.1%} Exp {m['exposure']:.1%}")
pd.DataFrame(rows).sort_values('sharpe', ascending=False)


## 5. Walk-Forward Validation (anti-trampa)
Nunca 2018-2026 -> optimizar -> reportar. Esquema:
```
2018-01-01 ----- 2020-12-31 | 2021-01-01 - 2021-12-31  fold1 IS 3y -> OOS 1y
2019-01-01 ----- 2021-12-31 | 2022-01-01 - 2022-12-31  fold2
2020-01-01 ----- 2022-12-31 | 2023-01-01 - 2023-12-31  fold3
2021-01-01 ----- 2023-12-31 | 2024-01-01 - 2024-12-31  fold4
2018-01-01 --------------- 2024-12-31 | 2025-01-01 - hoy  HOLDOUT final nunca visto
```
Solo IS ve optimizador, params congelados en OOS. Objetivo primario: Median OOS Sharpe.

In [ ]:
FOLDS=[{'train':('2018-01-01','2020-12-31'),'test':('2021-01-01','2021-12-31')},{'train':('2019-01-01','2021-12-31'),'test':('2022-01-01','2022-12-31')},{'train':('2020-01-01','2022-12-31'),'test':('2023-01-01','2023-12-31')},{'train':('2021-01-01','2023-12-31'),'test':('2024-01-01','2024-12-31')}]
FINAL_HOLDOUT=('2025-01-01', None)
# --- Ajuste dinamico al primer dato disponible (desconocido desde que ano) ---
try:
    _min_ts = df['timestamp'].min()
    _min_year = int(str(_min_ts)[:4])  # o _min_ts.year si es datetime
    if hasattr(_min_ts, 'year'):
        _min_year = _min_ts.year
    print(f'Data disponible desde {_min_ts} (ano {_min_year}) -> FOLDS se ajustaran si empieza antes de 2018')
    # Si datos empiezan en 2017, anadir fold extra 2017-2019->2020 o expandir train
    # Mantenemos FOLDS base pero train_start se adapta al min_year para no desperdiciar 2017
    # Sobrescribir train de cada fold para que empiece en _min_year si es <2018
    for _f in FOLDS:
        _orig_train_start = _f['train'][0]
        _train_y = int(_orig_train_start[:4])
        if _min_year < _train_y:
            _f['train'] = (f'{_min_year}-01-01', _f['train'][1])
    print('FOLDS ajustados:', FOLDS)
except Exception as _e:
    print('No se pudo ajustar FOLDS dinamico:', _e)

def slice_df(df, start, end):
    cond=pl.col('timestamp') >= pl.lit(start).str.to_datetime(time_zone='UTC')
    if end is not None:
        cond=cond & (pl.col('timestamp') <= pl.lit(end).str.to_datetime(time_zone='UTC'))
    return df.filter(cond)
for i,f in enumerate(FOLDS,1):
    tr=slice_df(df, f['train'][0], f['train'][1])
    te=slice_df(df, f['test'][0], f['test'][1])
    print(f'Fold {i}: train {f["train"]} -> {len(tr):,} | test {f["test"]} -> {len(te):,}')
hold=slice_df(df, FINAL_HOLDOUT[0], FINAL_HOLDOUT[1])
print(f'HOLDOUT {FINAL_HOLDOUT} -> {len(hold):,}')
def median_oos_sharpe(oos_metrics):
    return float(np.median([m['sharpe'] for m in oos_metrics])) if oos_metrics else -999


## 6. Optimizacion - TPE (Bayesian) y NSGA-II
Espacios: RSI period 2-50 OS 10-45 OB 55-90 OS<50<OB; MACD fast 3-30 slow 15-100 signal 2-30 Fast<Slow; MA fast 2-100 slow 20-400 Fast<Slow SMA/EMA separados; CCI period 5-50 entry -150..0 exit 0..150 entry<exit.

TPE: maximiza Median OOS Sharpe. NSGA-II: max Sharpe, max CAGR, min MaxDD, min Turnover. Filtros trades>=30, MaxDD>-50%, PF>1.

In [ ]:
try:
    import optuna
    from optuna.samplers import TPESampler, NSGAIISampler
    print('optuna', optuna.__version__)
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '--quiet'])
    import optuna
    from optuna.samplers import TPESampler, NSGAIISampler
    print('optuna instalado', optuna.__version__)
optuna.logging.set_verbosity(optuna.logging.WARNING)
def evaluate_params_on_df(df_is, df_oos, signal_fn, bt_kwargs):
    sig_is=signal_fn(df_is)
    sig_oos=signal_fn(df_oos)
    col_is=[c for c in sig_is.columns if c.startswith('signal')][0]
    col_oos=[c for c in sig_oos.columns if c.startswith('signal')][0]
    bt_is=backtest_long(sig_is, col_is, **bt_kwargs)
    bt_oos=backtest_long(sig_oos, col_oos, **bt_kwargs)
    m_is=summarize_bt(bt_is)
    m_oos=summarize_bt(bt_oos)
    return {'is':m_is,'oos':m_oos,'bt_is':bt_is,'bt_oos':bt_oos}
def is_valid_metrics(m, min_trades=30):
    if m['exposure'] < 0.005: return False
    if m['turnover'] < min_trades*0.5: return False
    if m['profit_factor'] < 1.0 and m['total_return'] <=0: return False
    if m['max_drawdown'] < -0.5: return False
    return True
print('Helpers listos. Demo RSI 14/30/70 fold1:')
df_is=slice_df(df,'2021-01-01','2023-12-31')
df_oos=slice_df(df,'2024-01-01','2024-12-31')
res=evaluate_params_on_df(df_is, df_oos, lambda d: signal_rsi_long(d,14,30,70), {'fee_bps':FEES_BPS,'slippage_bps':SLIPPAGE_BPS,'lag':LAG})
print('IS Sharpe', round(res['is']['sharpe'],2), 'OOS Sharpe', round(res['oos']['sharpe'],2), 'valid?', is_valid_metrics(res['oos']))


### 6.1 TPE - Median OOS Sharpe

In [ ]:
def run_tpe_optimization(indicator, df, folds=FOLDS, n_trials=80, seed=42):
    def objective(trial):
        if indicator=='rsi':
            period=trial.suggest_int('period',2,50)
            oversold=trial.suggest_int('oversold',10,45)
            overbought=trial.suggest_int('overbought',55,90)
            if not (oversold < 50 < overbought): raise optuna.TrialPruned()
            fn=lambda d: signal_rsi_long(d, period, oversold, overbought)
        elif indicator=='macd':
            fast=trial.suggest_int('fast',3,30)
            slow=trial.suggest_int('slow',15,100)
            signal=trial.suggest_int('signal',2,30)
            if not (fast < slow): raise optuna.TrialPruned()
            fn=lambda d: signal_macd_long(d, fast, slow, signal)
        elif indicator=='sma':
            fast=trial.suggest_int('fast',2,100)
            slow=trial.suggest_int('slow',20,400)
            if not (fast < slow): raise optuna.TrialPruned()
            fn=lambda d: signal_ma_long(d, fast, slow, 'sma')
        elif indicator=='ema':
            fast=trial.suggest_int('fast',2,100)
            slow=trial.suggest_int('slow',20,400)
            if not (fast < slow): raise optuna.TrialPruned()
            fn=lambda d: signal_ma_long(d, fast, slow, 'ema')
        elif indicator=='cci':
            period=trial.suggest_int('period',5,50)
            entry=trial.suggest_int('entry',-150,0)
            exit_=trial.suggest_int('exit',0,150)
            if not (entry < exit_): raise optuna.TrialPruned()
            fn=lambda d: signal_cci_long(d, period, entry, exit_)
        else: raise ValueError(indicator)
        oos_sharpes=[]
        for f in folds:
            df_is=slice_df(df, f['train'][0], f['train'][1])
            df_oos=slice_df(df, f['test'][0], f['test'][1])
            try:
                res=evaluate_params_on_df(df_is, df_oos, fn, {'fee_bps':FEES_BPS,'slippage_bps':SLIPPAGE_BPS,'lag':LAG})
                m_oos=res['oos']
                oos_sharpes.append(m_oos['sharpe'] if is_valid_metrics(m_oos) else -5.0)
            except: oos_sharpes.append(-5.0)
        median=float(np.median(oos_sharpes)) if oos_sharpes else -5.0
        trial.set_user_attr('oos_sharpes', oos_sharpes)
        return median
    study=optuna.create_study(direction='maximize', sampler=TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study
for ind in ['rsi','macd','sma','ema','cci']:
    print(f"\n=== Demo TPE {ind} n_trials=10 ===")
    st=run_tpe_optimization(ind, df, n_trials=10, seed=SEED)
    print(f"Best median Sharpe {st.best_value:.3f} params={st.best_params}")


### 6.2 NSGA-II - Pareto (4 objetivos)

In [ ]:
def run_nsga_optimization(indicator, df, folds=FOLDS, n_trials=80, seed=42):
    def objective(trial):
        if indicator=='rsi':
            period=trial.suggest_int('period',2,50)
            oversold=trial.suggest_int('oversold',10,45)
            overbought=trial.suggest_int('overbought',55,90)
            if not (oversold < 50 < overbought): raise optuna.TrialPruned()
            fn=lambda d: signal_rsi_long(d, period, oversold, overbought)
        elif indicator=='macd':
            fast=trial.suggest_int('fast',3,30)
            slow=trial.suggest_int('slow',15,100)
            signal=trial.suggest_int('signal',2,30)
            if not (fast < slow): raise optuna.TrialPruned()
            fn=lambda d: signal_macd_long(d, fast, slow, signal)
        elif indicator=='sma':
            fast=trial.suggest_int('fast',2,100)
            slow=trial.suggest_int('slow',20,400)
            if not (fast < slow): raise optuna.TrialPruned()
            fn=lambda d: signal_ma_long(d, fast, slow, 'sma')
        elif indicator=='ema':
            fast=trial.suggest_int('fast',2,100)
            slow=trial.suggest_int('slow',20,400)
            if not (fast < slow): raise optuna.TrialPruned()
            fn=lambda d: signal_ma_long(d, fast, slow, 'ema')
        elif indicator=='cci':
            period=trial.suggest_int('period',5,50)
            entry=trial.suggest_int('entry',-150,0)
            exit_=trial.suggest_int('exit',0,150)
            if not (entry < exit_): raise optuna.TrialPruned()
            fn=lambda d: signal_cci_long(d, period, entry, exit_)
        else: raise ValueError
        oos_list=[]
        for f in folds:
            df_is=slice_df(df, f['train'][0], f['train'][1])
            df_oos=slice_df(df, f['test'][0], f['test'][1])
            try:
                res=evaluate_params_on_df(df_is, df_oos, fn, {'fee_bps':FEES_BPS,'slippage_bps':SLIPPAGE_BPS,'lag':LAG})
                oos_list.append(res['oos'])
            except:
                oos_list.append({'sharpe':-5,'cagr':-1,'max_drawdown':-1,'turnover':9999})
        med_sharpe=float(np.median([m['sharpe'] for m in oos_list]))
        med_cagr=float(np.median([m['cagr'] for m in oos_list]))
        med_mdd=float(np.median([m['max_drawdown'] for m in oos_list]))
        med_turn=float(np.median([m['turnover'] for m in oos_list]))
        return med_sharpe, med_cagr, -med_mdd, -med_turn
    study=optuna.create_study(directions=['maximize','maximize','maximize','maximize'], sampler=NSGAIISampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study
print('=== Demo NSGA-II rsi 10 trials ===')
st_nsga=run_nsga_optimization('rsi', df, n_trials=10, seed=SEED)
print(f'NSGA trials {len(st_nsga.trials)} Pareto {len(st_nsga.best_trials)}')
for t in st_nsga.best_trials[:3]:
    print(t.number, t.values, t.params)


## 7. Experimentos Aislados - Ejecucion Completa TPE
5 experimentos (RSI, MACD, SMA, EMA, CCI) con N_TRIALS_TPE trials. Con 1h ~5*80*4=1600 backtests ~5-15 min. Con 1m horas.

In [ ]:
RESULTS_TPE={}
for indicator in ['rsi','macd','sma','ema','cci']:
    print(f"\n{'='*60}\n[ TPE ] {indicator.upper()} trials={N_TRIALS_TPE} tf={TIMEFRAME}\n{'='*60}")
    study=run_tpe_optimization(indicator, df, n_trials=N_TRIALS_TPE, seed=SEED)
    best=study.best_params
    best_median=study.best_value
    if indicator=='rsi': fn_best=lambda d, p=best: signal_rsi_long(d, p['period'], p['oversold'], p['overbought'])
    elif indicator=='macd': fn_best=lambda d, p=best: signal_macd_long(d, p['fast'], p['slow'], p['signal'])
    elif indicator=='sma': fn_best=lambda d, p=best: signal_ma_long(d, p['fast'], p['slow'], 'sma')
    elif indicator=='ema': fn_best=lambda d, p=best: signal_ma_long(d, p['fast'], p['slow'], 'ema')
    elif indicator=='cci': fn_best=lambda d, p=best: signal_cci_long(d, p['period'], p['entry'], p['exit'])
    fold_rows=[]
    for f in FOLDS:
        df_is=slice_df(df, f['train'][0], f['train'][1])
        df_oos=slice_df(df, f['test'][0], f['test'][1])
        res=evaluate_params_on_df(df_is, df_oos, fn_best, {'fee_bps':FEES_BPS,'slippage_bps':SLIPPAGE_BPS,'lag':LAG})
        fold_rows.append({'fold': f"{f['train'][0]}->{f['train'][1]} | {f['test'][0]}->{f['test'][1]}", **{f"oos_{k}":v for k,v in res['oos'].items()}, **{f"is_{k}":v for k,v in res['is'].items()}})
    df_train_full=slice_df(df,'2018-01-01','2024-12-31')
    df_hold=slice_df(df,FINAL_HOLDOUT[0], None)
    sig_hold=fn_best(df_hold)
    col_hold=[c for c in sig_hold.columns if c.startswith('signal')][0]
    bt_hold=backtest_long(sig_hold, col_hold, fee_bps=FEES_BPS, slippage_bps=SLIPPAGE_BPS, lag=LAG)
    m_hold=summarize_bt(bt_hold)
    sig_is_full=fn_best(df_train_full)
    col_is_full=[c for c in sig_is_full.columns if c.startswith('signal')][0]
    bt_is_full=backtest_long(sig_is_full, col_is_full, fee_bps=FEES_BPS, slippage_bps=SLIPPAGE_BPS, lag=LAG)
    m_is_full=summarize_bt(bt_is_full)
    RESULTS_TPE[indicator]={'best_params':best,'median_oos_sharpe':best_median,'fold_rows':fold_rows,'m_is_full':m_is_full,'m_hold':m_hold,'study':study,'bt_hold':bt_hold,'bt_is_full':bt_is_full}
    print(f"-> {indicator.upper()} best {best} median OOS Sharpe {best_median:.3f}")
    print(f"  IS 2018-2024 Sharpe {m_is_full['sharpe']:.2f} CAGR {m_is_full['cagr']:.1%} MaxDD {m_is_full['max_drawdown']:.1%}")
    print(f"  HOLDOUT 2025-2026 Sharpe {m_hold['sharpe']:.2f} CAGR {m_hold['cagr']:.1%} MaxDD {m_hold['max_drawdown']:.1%} PF {m_hold['profit_factor']:.2f}")
import pickle
(REPO / 'data/processed').mkdir(exist_ok=True)
with open(REPO / 'data/processed/tpe_results.pkl','wb') as f:
    pickle.dump({k: {kk:v for kk,v in d.items() if kk not in ['study','bt_hold','bt_is_full']} for k,d in RESULTS_TPE.items()}, f)
print('\nGuardado data/processed/tpe_results.pkl')
rows=[]
for ind, d in RESULTS_TPE.items():
    rows.append({'indicator': ind.upper(), 'best_params': str(d['best_params']), 'median_oos_sharpe': d['median_oos_sharpe'], 'is_sharpe': d['m_is_full']['sharpe'], 'hold_sharpe': d['m_hold']['sharpe'], 'hold_cagr': d['m_hold']['cagr'], 'hold_mdd': d['m_hold']['max_drawdown'], 'hold_pf': d['m_hold']['profit_factor'], 'hold_wr': d['m_hold']['win_rate']})
pd.DataFrame(rows).sort_values('median_oos_sharpe', ascending=False)


### 7.1 CCI como filtro (no direccional)
CCI_t -> |Return_{t+n}| y volatilidad. Si CCI alto predice movimientos grandes, usar como filtro: solo operar cuando CCI > percentile 70.

In [ ]:
df_cci=df.with_columns(cci(pl.col('high'), pl.col('low'), pl.col('close'),20).alias('cci'))
for n in [5,15,60]:
    df_cci=df_cci.with_columns((pl.col('close').shift(-n)/pl.col('close')-1).abs().alias(f'abs_ret_{n}'))
from scipy.stats import spearmanr
sub=df_cci.filter(pl.col('timestamp') >= pl.lit('2023-01-01').str.to_datetime(time_zone='UTC')).drop_nulls(subset=['cci','abs_ret_60'])
cc, p = spearmanr(sub['cci'].to_numpy(), sub['abs_ret_60'].to_numpy())
print(f'Spearman CCI(20) vs |return_60| (2023+): rho={cc:.3f} p={p:.2e} n={len(sub)}')
sub2=df_cci.with_columns((pl.col('close').log() - pl.col('close').shift(1).log()).alias('_lr'))
sub2=sub2.with_columns(pl.col('_lr').rolling_std(60).alias('real_vol_60'))
sub2=sub2.with_columns(pl.col('real_vol_60').shift(-60).alias('fwd_vol'))
sub3=sub2.filter(pl.col('timestamp') >= pl.lit('2023-01-01').str.to_datetime(time_zone='UTC')).drop_nulls(subset=['cci','fwd_vol'])
cc2, p2 = spearmanr(sub3['cci'].to_numpy(), sub3['fwd_vol'].to_numpy())
print(f'Spearman CCI(20) vs fwd_vol_60: rho={cc2:.3f} p={p2:.2e}')


## 8. Metricas Completas (17 categorias)
Retorno: total_return, cagr; Riesgo: max_drawdown, mdd_duration, vol, VaR95, Ulcer; Ajustadas: sharpe, sortino, calmar, deflated_sharpe; Por operacion: n_trades, win_rate, avg_win/loss, payoff, profit_factor, expectancy; Costos: fees_total, turnover, exposure, break_even_cost; Estadisticas: skew, kurtosis, rolling_sharpe, param_stability

In [ ]:
def expand_metrics(bt):
    base=summarize_bt(bt)
    rets=bt['strategy_ret'].drop_nulls().to_numpy()
    eq=bt['equity'].to_numpy()
    peak=np.maximum.accumulate(eq)
    dd=(eq-peak)/peak*100
    ulcer=float(np.sqrt(np.mean(dd**2)))
    var95=float(np.percentile(rets, 5))
    cvar95=float(rets[rets <= var95].mean()) if (rets <= var95).any() else var95
    n_trades=int(bt['_turnover'].sum()/2) if '_turnover' in bt.columns else 0
    be=float(base['total_return']/base['turnover']) if base['turnover']>0 else 0.0
    consec=0; max_consec=0
    for r in rets:
        if r<0: consec+=1; max_consec=max(max_consec, consec)
        else: consec=0
    base.update({'ulcer':ulcer,'var95':var95,'cvar95':cvar95,'n_trades':n_trades,'break_even_cost':be,'max_consec_losses':max_consec})
    return base
for ind, d in RESULTS_TPE.items():
    m=expand_metrics(d['bt_hold'])
    print(f"{ind.upper():4s} holdout -> n_trades {m['n_trades']} BEcost {m['break_even_cost']:.4f} ({m['break_even_cost']*10000:.0f} bps) Ulcer {m['ulcer']:.1f}% VaR95 {m['var95']:.3%}")
fig, ax=plt.subplots(figsize=(12,5))
hold_example=list(RESULTS_TPE.values())[0]['bt_hold']
ax.plot(hold_example['timestamp'].to_list(), hold_example['bh_equity'].to_list(), label='Buy & Hold', lw=1.5, ls='--', color='black')
for ind, d in RESULTS_TPE.items():
    bt=d['bt_hold']
    ax.plot(bt['timestamp'].to_list(), bt['equity'].to_list(), label=ind.upper(), lw=1.0)
ax.set_title(f'Equity HOLDOUT 2025-2026 ({TIMEFRAME}, long-only, fees {FEES_BPS}bps+{SLIPPAGE_BPS}bps)')
ax.set_ylabel('Equity (10k)'); ax.legend(); plt.tight_layout(); plt.show()
fig, ax=plt.subplots(figsize=(12,3))
for ind, d in RESULTS_TPE.items():
    bt=d['bt_hold']
    rets=bt['strategy_ret'].fill_null(0).to_numpy()
    win=720 if TIMEFRAME=='1h' else 100
    roll=[np.mean(rets[max(0,i-win):i])/np.std(rets[max(0,i-win):i])*np.sqrt(365*24) if i>win and np.std(rets[max(0,i-win):i])>0 else 0 for i in range(len(rets))]
    ax.plot(bt['timestamp'].to_list(), roll, label=ind.upper(), lw=0.8, alpha=0.9)
ax.axhline(0,c='k',lw=0.7); ax.set_title('Rolling Sharpe (30d) - HOLDOUT'); ax.legend(ncol=5, fontsize=8); plt.show()


### 8.2 Equity desde el mínimo (no solo HOLDOUT)
> Holdout 2025-2026 es útil OOS puro, pero también graficamos **equity completo desde `df['timestamp'].min()` (2017-08-17)** con los *best params* (median OOS Sharpe). Así se ve robustez entre halvings y si el edge existía desde el inicio.


In [ ]:
# --- Equity FULL desde el minimo con best params (vs solo HOLDOUT) ---
fig, ax = plt.subplots(figsize=(12,5))
# BH full para referencia
df_full_bh = df.clone()  # df es el resampleado 4h desde 2017
# Construir benchmark BH sobre df completo (no solo hold)
# Usamos bt de un indicador cualquiera para extraer bh_equity escalado a todo el rango
# Mejor: calcular BH equity directamente desde close
# Primero: equity BH full
close_full = df['close'].to_numpy()
bh_growth_full = close_full / close_full[0]
bh_equity_full = bh_growth_full * 10000
ax.plot(df['timestamp'].to_list(), bh_equity_full, label='Buy & Hold (full)', lw=1.5, ls='--', color='black')
for ind, d in RESULTS_TPE.items():
    best = d['best_params']
    # reconstruir fn_best
    if ind=='rsi': fn=lambda x, p=best: signal_rsi_long(x, p['period'], p['oversold'], p['overbought'])
    elif ind=='macd': fn=lambda x, p=best: signal_macd_long(x, p['fast'], p['slow'], p['signal'])
    elif ind=='sma': fn=lambda x, p=best: signal_ma_long(x, p['fast'], p['slow'], 'sma')
    elif ind=='ema': fn=lambda x, p=best: signal_ma_long(x, p['fast'], p['slow'], 'ema')
    elif ind=='cci': fn=lambda x, p=best: signal_cci_long(x, p['period'], p['entry'], p['exit'])
    sig_full = fn(df)
    col = [c for c in sig_full.columns if c.startswith('signal')][0]
    bt_full = backtest_long(sig_full, col, fee_bps=FEES_BPS, slippage_bps=SLIPPAGE_BPS, lag=LAG)
    ax.plot(bt_full['timestamp'].to_list(), bt_full['equity'].to_list(), label=ind.upper(), lw=0.9, alpha=0.9)
    # Guardar para tabla por ciclo si se quiere
ax.set_title(f'Equity FULL desde {str(df["timestamp"].min())[:10]} ({TIMEFRAME}, long-only, fees {FEES_BPS}bps+{SLIPPAGE_BPS}bps) - best params median OOS')
ax.set_ylabel('Equity (10k)'); ax.set_xlabel(f'Desde minimo {str(df["timestamp"].min())[:10]} -> {str(df["timestamp"].max())[:10]}')
ax.legend(fontsize=7, ncol=2); plt.tight_layout()
# Halvings
try:
    add_halvings_to_ax(ax)
    plt.show()
except: plt.show()
# Tambien log scale para ver CAGR largo plazo
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(df['timestamp'].to_list(), bh_equity_full, label='Buy & Hold', lw=1.2, ls='--', color='black')
for ind, d in RESULTS_TPE.items():
    best = d['best_params']
    if ind=='rsi': fn=lambda x, p=best: signal_rsi_long(x, p['period'], p['oversold'], p['overbought'])
    elif ind=='macd': fn=lambda x, p=best: signal_macd_long(x, p['fast'], p['slow'], p['signal'])
    elif ind=='sma': fn=lambda x, p=best: signal_ma_long(x, p['fast'], p['slow'], 'sma')
    elif ind=='ema': fn=lambda x, p=best: signal_ma_long(x, p['fast'], p['slow'], 'ema')
    elif ind=='cci': fn=lambda x, p=best: signal_cci_long(x, p['period'], p['entry'], p['exit'])
    sig_full = fn(df)
    col = [c for c in sig_full.columns if c.startswith('signal')][0]
    bt_full = backtest_long(sig_full, col, fee_bps=FEES_BPS, slippage_bps=SLIPPAGE_BPS, lag=LAG)
    ax.plot(bt_full['timestamp'].to_list(), bt_full['equity'].to_list(), label=ind.upper(), lw=0.9)
ax.set_yscale('log'); ax.set_title(f'Equity FULL log-scale desde {str(df["timestamp"].min())[:10]}')
ax.set_ylabel('Equity log'); ax.legend(fontsize=7); plt.tight_layout()
try: add_halvings_to_ax(ax)
except: pass
plt.show()
print(f'Graficado FULL {len(df):,} velas {TIMEFRAME} desde {df["timestamp"].min()} -> {df["timestamp"].max()} (holdout era solo {len(hold):,} velas 2025-2026)')


### 8.1 Deflated Sharpe & Parameter Stability
Deflated Sharpe corrige por N trials. Estabilidad: si best=RSI 17 Sharpe 1.82 aislado y vecinos 16/18 caen a 0.4, descartar. Plateau 14->1.31,15->1.38,16->1.42,17->1.45 es creible. Guardar todos los trials.

In [ ]:
try:
    study_rsi=RESULTS_TPE['rsi']['study']
    df_trials=study_rsi.trials_dataframe()
    fig, ax=plt.subplots(figsize=(6,3))
    ax.scatter(df_trials['params_period'], df_trials['value'], alpha=0.6, s=12)
    ax.set_xlabel('RSI period'); ax.set_ylabel('Median OOS Sharpe'); ax.set_title('RSI - stability (period)')
    plt.show()
    fig, ax=plt.subplots(figsize=(5,4))
    sc=ax.scatter(df_trials['params_oversold'], df_trials['params_overbought'], c=df_trials['value'], cmap='RdYlGn', s=18)
    ax.set_xlabel('oversold'); ax.set_ylabel('overbought'); ax.set_title('RSI - OS vs OB (color=Sharpe)')
    plt.colorbar(sc, label='median Sharpe'); plt.show()
except Exception as e:
    print('No hay study RSI aun:', e)
import math
def deflated_sharpe(sharpe, n_trials, n_obs):
    if n_obs<=1: return sharpe
    return sharpe - math.sqrt(2*math.log(n_trials)/n_obs)
for ind, d in RESULTS_TPE.items():
    m=d['m_hold']
    ds=deflated_sharpe(m['sharpe'], N_TRIALS_TPE*len(FOLDS), d['bt_hold'].height)
    print(f"{ind.upper():4s} Sharpe {m['sharpe']:.2f} -> Deflated {ds:.2f} (N={N_TRIALS_TPE*len(FOLDS)}, T={d['bt_hold'].height})")


## 9. Comparacion Final vs Buy & Hold
Todo vs BTC Buy & Hold mismo periodo. Si BH CAGR 40% MaxDD -30% y estrategia CAGR 15% MaxDD -25%, no aporta alpha.

In [ ]:
rows=[]
for ind, d in RESULTS_TPE.items():
    for period_name, m in [('IS 2018-2024', d['m_is_full']), ('HOLDOUT 2025-2026', d['m_hold'])]:
        rows.append({'indicator': ind.upper(), 'period': period_name, 'CAGR': m['cagr'], 'Sharpe': m['sharpe'], 'Sortino': m['sortino'], 'Calmar': m['calmar'], 'MaxDD': m['max_drawdown'], 'TotalRet': m['total_return'], 'PF': m['profit_factor'], 'WR': m['win_rate'], 'Exposure': m['exposure'], 'Turnover': m['turnover'], 'BH_CAGR': m['bh_cagr'], 'BH_Sharpe': m['bh_sharpe'], 'ActiveReturn': m['cagr'] - m['bh_cagr']})
bh_row_is=RESULTS_TPE['rsi']['m_is_full']
bh_row_hold=RESULTS_TPE['rsi']['m_hold']
for period_name, m in [('IS 2018-2024', bh_row_is), ('HOLDOUT 2025-2026', bh_row_hold)]:
    rows.append({'indicator':'BUY & HOLD','period':period_name,'CAGR':m['bh_cagr'],'Sharpe':m['bh_sharpe'],'Sortino':np.nan,'Calmar':m['bh_cagr']/abs(m['max_drawdown']) if m['max_drawdown']!=0 else np.nan,'MaxDD':m['max_drawdown'],'TotalRet':m['total_return'],'PF':np.nan,'WR':np.nan,'Exposure':1.0,'Turnover':0,'BH_CAGR':m['bh_cagr'],'BH_Sharpe':m['bh_sharpe'],'ActiveReturn':0})
comp=pd.DataFrame(rows)
print('=== IS 2018-2024 ===')
print(comp[comp['period']=='IS 2018-2024'].sort_values('Sharpe', ascending=False).to_string(index=False))
print('\n=== HOLDOUT 2025-2026 (OOS puro) ===')
print(comp[comp['period']=='HOLDOUT 2025-2026'].sort_values('Sharpe', ascending=False).to_string(index=False))
sub=comp[comp['period']=='HOLDOUT 2025-2026']
fig, ax=plt.subplots(figsize=(6,4))
for _,r in sub.iterrows():
    ax.scatter(abs(r['MaxDD'])*100, r['CAGR']*100, s=80, label=r['indicator'])
    ax.annotate(r['indicator'], (abs(r['MaxDD'])*100, r['CAGR']*100), textcoords='offset points', xytext=(5,5), fontsize=8)
ax.set_xlabel('|MaxDD| %'); ax.set_ylabel('CAGR %'); ax.set_title('HOLDOUT: CAGR vs Riesgo'); ax.legend(fontsize=7); plt.grid(True, alpha=0.3); plt.show()


## 10. Conclusiones y Proximos Pasos
Que responder tras 7: 1) Algun indicador supera BH en Median OOS Sharpe y CAGR neto? 2) Degradacion IS->OOS? Si >30% caida -> sobreajuste. 3) Plateau estable o pico aislado? 4) HOLDOUT confirma edge? 5) CCI como filtro mejora SMA/EMA?

Proximos experimentos: SMA vs EMA test Diebold-Mariano, NSGA-II Pareto Sharpe vs Turnover, taker_buy imbalance + RSI filtro, Monte Carlo shuffling.

Reproducibilidad: jupyter nbconvert --to notebook --execute notebooks/05_experimento_indicadores_aislados.ipynb --output 05_executed.ipynb
Para 1m final cambiar TIMEFRAME="1m" y N_TRIALS_TPE=100 en seccion 1.

Entregable: este notebook es Experiment 01. Experiment 02 combinara RSI+CCI filtro solo si indicador aislado muestra edge OOS creible.

In [ ]:
summary_path=REPO / 'docs/experimento_01_resumen.md'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write('# Experimento 01 - Resumen (auto-generado)\n\n')
    f.write(f'Fecha: {dt.datetime.now().isoformat()}  |  TIMEFRAME={TIMEFRAME}  |  Trials TPE={N_TRIALS_TPE}\n\n')
    f.write('| Indicador | Best Params | Median OOS Sharpe | IS Sharpe | HOLDOUT Sharpe | HOLDOUT CAGR | MaxDD | PF |\n')
    f.write('|-----------|-------------|-------------------|-----------|---------------|-------------|-------|----|\n')
    for ind, d in RESULTS_TPE.items():
        f.write(f"| {ind.upper()} | `{d['best_params']}` | {d['median_oos_sharpe']:.3f} | {d['m_is_full']['sharpe']:.2f} | {d['m_hold']['sharpe']:.2f} | {d['m_hold']['cagr']:.1%} | {d['m_hold']['max_drawdown']:.1%} | {d['m_hold']['profit_factor']:.2f} |\n")
    f.write('\n> Generado desde notebooks/05_experimento_indicadores_aislados.ipynb\n')
print(f'Resumen guardado en {summary_path}')
print(open(summary_path, encoding='utf-8').read())


In [ ]:
print('Imports OK, df:', df.shape, 'TIMEFRAME', TIMEFRAME)
print('Tests:', 'RSI' in str(signal_rsi_long), 'CCI' in str(signal_cci_long))
print('Backtester long-only:', backtest_long.__doc__[:60] if backtest_long.__doc__ else 'ok')
print('Optuna:', optuna.__version__)
print('Parquet:', (REPO / 'data/processed/btcusdt_1m.parquet').stat().st_size/1e6, 'MB')
print('AGENTS.md:', Path('AGENTS.md').exists())
